In [2]:
# =============================================================
# Modul 0: Datenvorbereitung / Data Preparation
# Projekt: RetailCo Financial Analysis
# Datensatz: Bike Sales in Europe (Kaggle, Sadiq Shah)
# Autor: Yurii Oleshchuk
# Datum: 2026
# =============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------
# 1. DATEN LADEN / LOAD DATA
# DE: Rohdaten aus dem CSV-Datei laden
# EN: Load raw data from CSV file
# ------------------------------------------------------------------

df = pd.read_csv('../data/raw/bike_sales.csv', encoding='latin1')

print("Shape:", df.shape)
print(df.dtypes)
print(df.head(3))

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/bike_sales.csv'

In [ ]:
# ------------------------------------------------------------------
# 2. DATENQUALITÄT PRÜFEN / DATA QUALITY CHECK
# DE: Fehlende Werte, Duplikate und Ausreißer identifizieren
# EN: Identify missing values, duplicates and outliers
# ------------------------------------------------------------------

print("=== Fehlende Werte / Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplikate / Duplicates ===")
print(f"Duplikate gefunden: {df.duplicated().sum()}")

print("\n=== Deskriptive Statistik / Descriptive Statistics ===")
print(df[['Revenue', 'Cost', 'Profit', 'Order_Quantity', 
          'Unit_Price', 'Unit_Cost']].describe().round(2))

In [ ]:
# ------------------------------------------------------------------
# 3. DATENTRANSFORMATION / DATA TRANSFORMATION
# DE: Spalten umbenennen, Datentypen korrigieren, neue Spalten anlegen
# EN: Rename columns, fix data types, create new columns
# ------------------------------------------------------------------

# Spaltennamen vereinheitlichen / Standardize column names
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

# Datum parsen / Parse date
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')

# Neue Spalten / New columns
df['year_month']    = df['date'].dt.to_period('M')   # für Monatsabschluss
df['quarter']       = df['date'].dt.quarter.map(lambda q: f'Q{q}')
df['fiscal_year']   = df['date'].dt.year

# Marge berechnen / Calculate margin
df['gross_margin_pct'] = (df['profit'] / df['revenue'] * 100).round(2)

# Prüfen / Check
print(df[['date','year_month','quarter','fiscal_year',
          'revenue','cost','profit','gross_margin_pct']].head(5))

In [ ]:
# ------------------------------------------------------------------
# 4. LÄNDER-MAPPING (DE + EN)
# DE: Ländernamen auf Deutsch und Englisch hinzufügen
# EN: Add country names in German and English
# ------------------------------------------------------------------

country_map_de = {
    'Australia':      'Australien',
    'Canada':         'Kanada',
    'France':         'Frankreich',
    'Germany':        'Deutschland',
    'United Kingdom': 'Vereinigtes Königreich',
    'United States':  'Vereinigte Staaten'
}

df['land_de'] = df['country'].map(country_map_de)  # Deutsch
df['land_en'] = df['country']                       # English (Original)

print(df[['country','land_de']].value_counts())

In [ ]:
# ------------------------------------------------------------------
# 5. BUDGET-SPALTEN GENERIEREN / GENERATE BUDGET COLUMNS
# DE: Plan-Werte simulieren (±12% Abweichung vom Ist-Wert)
#     → Standard-Methode im Controlling für historische Projekte
# EN: Simulate plan values (±12% deviation from actual)
#     → Standard method in Controlling for historical datasets
# ------------------------------------------------------------------

np.random.seed(42)  # Reproduzierbarkeit / Reproducibility

n = len(df)
variation = np.random.uniform(-0.12, 0.12, n)

df['revenue_plan'] = (df['revenue'] * (1 + variation)).round(0)
df['cost_plan']    = (df['cost']    * (1 + variation * 0.8)).round(0)
df['profit_plan']  = (df['revenue_plan'] - df['cost_plan']).round(0)

# Abweichungen / Variances
df['revenue_var_abs'] = df['revenue'] - df['revenue_plan']
df['revenue_var_pct'] = (df['revenue_var_abs'] / df['revenue_plan'] * 100).round(2)

print("Budget-Spalten erstellt / Budget columns created:")
print(df[['revenue','revenue_plan','revenue_var_abs','revenue_var_pct']].head(5))

In [ ]:
# ------------------------------------------------------------------
# 6. BEREINIGTE DATEN SPEICHERN / SAVE CLEANED DATA
# DE: Verarbeitete Daten als CSV speichern
# EN: Save processed data as CSV
# ------------------------------------------------------------------

df.to_csv('../data/processed/bike_sales_clean.csv', index=False, encoding='utf-8-sig')

print(f"✓ Datei gespeichert / File saved: bike_sales_clean.csv")
print(f"✓ Zeilen / Rows: {len(df):,}")
print(f"✓ Spalten / Columns: {len(df.columns)}")
print(f"\nFinale Spalten / Final columns:")
print(df.columns.tolist())